In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split


def compute_feature_path_diversity(X, y, n_models=10):
    """
    Entraîne plusieurs modèles sur des sous-échantillons et mesure si les
    features importantes changent d'un modèle à l'autre (Chemins multiples)
    ou restent fixes (Vérité unique).
    """
    feature_importances = []

    for _ in range(n_models):
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)
        model = RandomForestClassifier(n_estimators=50, max_features="sqrt")
        model.fit(X_train, y_train)

        # Un modèle doit être performant pour que son explication compte
        if model.score(X_test, y_test) > 0.75:
            feature_importances.append(model.feature_importances_)

    if not feature_importances:
        return 0.0

    importances_matrix = np.array(feature_importances)

    # Calcule la variance de l'importance pour chaque feature, puis la moyenne
    # Une variance élevée = les modèles utilisent des features différentes (Rouge sur votre heatmap)
    # Une variance faible = les modèles utilisent toujours les mêmes features (Bleu sur votre heatmap)
    diversity_score = np.mean(np.var(importances_matrix, axis=0))

    return diversity_score